In [ ]:
!pip install kafka-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.3/326.3 kB 8.2 MB/s eta 0:00:00


In [1]:
from kafka import KafkaProducer
import json

ModuleNotFoundError: No module named 'kafka'

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/Colab Notebooks/NFV3DATA-A11964_A11964.zip'
extract_path = '/content/dataset'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

os.listdir(extract_path)

['f7546561558c07c5_NFV3DATA-A11964_A11964']

In [5]:
base_path = '/content/dataset'
for root, dirs, files in os.walk(base_path):
    for file in files:
        print(os.path.join(root, file))


/content/dataset/f7546561558c07c5_NFV3DATA-A11964_A11964/manifest-sha1.txt
/content/dataset/f7546561558c07c5_NFV3DATA-A11964_A11964/bag-info.txt
/content/dataset/f7546561558c07c5_NFV3DATA-A11964_A11964/tagmanifest-sha1.txt
/content/dataset/f7546561558c07c5_NFV3DATA-A11964_A11964/FurtherInformation.txt
/content/dataset/f7546561558c07c5_NFV3DATA-A11964_A11964/bagit.txt
/content/dataset/f7546561558c07c5_NFV3DATA-A11964_A11964/data/NetFlow_v3_Features.csv
/content/dataset/f7546561558c07c5_NFV3DATA-A11964_A11964/data/NF-UNSW-NB15-v3.csv


In [6]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

In [ ]:
file_path = "/content/dataset/f7546561558c07c5_NFV3DATA-A11964_A11964/data/NF-UNSW-NB15-v3.csv"

df = pd.read_csv(file_path)

In [ ]:
print(df.columns.tolist())

['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR', 'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE', 'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE', 'SRC_TO_DST_IAT_MIN', 'SRC_TO_DST_IAT_MAX', 'SRC_TO_DST_IAT_AVG', 'SRC_TO_DST_IAT_STDDEV', 

In [ ]:
print("--- SUMMARY OF KEPT LOGS ---")
print(f"Total Logs Kept: {len(df)}")
print("\nBreakdown by Label:")
print(df['Label'].value_counts())
print("\nBreakdown by Attack Category:")
print(df['Attack'].value_counts())

--- SUMMARY OF KEPT LOGS ---
Total Logs Kept: 2365424

Breakdown by Label:
Label
0    2237731
1     127693
Name: count, dtype: int64

Breakdown by Attack Category:
Attack
Benign            2237731
Exploits            42748
Fuzzers             33816
Generic             19651
Reconnaissance      17074
DoS                  5980
Backdoor             4659
Shellcode            2381
Analysis             1226
Worms                 158
Name: count, dtype: int64


In [ ]:
df = df.fillna(0)
df['FLOW_START_MILLISECONDS'] = pd.to_datetime(df['FLOW_START_MILLISECONDS'], unit='ms')
df['START_HOUR'] = df['FLOW_START_MILLISECONDS'].dt.hour

columns_to_drop = [
    'FLOW_START_MILLISECONDS',
    'FLOW_END_MILLISECONDS',
]

df_refined = df.drop(columns=columns_to_drop, errors='ignore')


In [ ]:
print(df.columns.tolist())

['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR', 'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE', 'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE', 'SRC_TO_DST_IAT_MIN', 'SRC_TO_DST_IAT_MAX', 'SRC_TO_DST_IAT_AVG', 'SRC_TO_DST_IAT_STDDEV', 

In [ ]:
df_refined['hour_sin'] = np.sin(2 * np.pi * df_refined['START_HOUR'] / 24)
df_refined['hour_cos'] = np.cos(2 * np.pi * df_refined['START_HOUR'] / 24)

df_refined = df_refined.drop(columns=['START_HOUR'])
save_path = "/content/drive/MyDrive/Colab Notebooks/df_refined.csv"
df_refined.to_csv(save_path, index=False)

In [ ]:
print(df_refined.columns.tolist())

In [ ]:
''''import hashlib

df_refined['log_id'] = df_refined.apply(lambda x: hashlib.sha256(str(x).encode()).hexdigest(), axis=1)

identity_features = ['IPV4_SRC_ADDR', 'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'log_id']
target_features = ['Label', 'Attack']

numerical_features = [
    'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS',
    'FLOW_DURATION_MILLISECONDS', 'SRC_TO_DST_IAT_AVG', 'DST_TO_SRC_IAT_AVG'
]

for col in numerical_features:
    df_refined[col] = np.log1p(df_refined[col])
print("Refinement Complete. Forensic hashes generated and data normalized.")'''

'\'import hashlib\n\ndf_refined[\'log_id\'] = df_refined.apply(lambda x: hashlib.sha256(str(x).encode()).hexdigest(), axis=1)\n\nidentity_features = [\'IPV4_SRC_ADDR\', \'L4_SRC_PORT\', \'IPV4_DST_ADDR\', \'L4_DST_PORT\', \'log_id\']\ntarget_features = [\'Label\', \'Attack\']\n\nnumerical_features = [\n    \'IN_BYTES\', \'IN_PKTS\', \'OUT_BYTES\', \'OUT_PKTS\',\n    \'FLOW_DURATION_MILLISECONDS\', \'SRC_TO_DST_IAT_AVG\', \'DST_TO_SRC_IAT_AVG\'\n]\n\nfor col in numerical_features:\n    df_refined[col] = np.log1p(df_refined[col])\nprint("Refinement Complete. Forensic hashes generated and data normalized.")'

In [ ]:
package = joblib.load('/content/drive/MyDrive/Colab Notebooks/gatekeeper_system.pkl')

gatekeeper = package['model']
gate_threshold = package['threshold']
trained_features = package['features']

print("Gatekeeper system loaded and ready for inference!")


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/gatekeeper_system.pkl'

In [ ]:
X_new = df_refined.drop(columns=['Label', 'Attack'], errors='ignore')

X_new = X_new[trained_features]

new_scores = gatekeeper.decision_function(X_new)

decisions = np.where(new_scores <= gate_threshold, -1, 1)

df_new_audit = df_refined.copy()
df_new_audit['Gatekeeper_Decision'] = decisions

kept_logs = df_new_audit[df_new_audit['Gatekeeper_Decision'] == -1].copy()
dropped_logs = df_new_audit[df_new_audit['Gatekeeper_Decision'] == 1].copy()

total = len(df_refined)
savings = (len(dropped_logs) / total) * 100

print(f"--- INFERENCE REPORT ---")
print(f"Total New Logs Processed: {total}")
print(f"Logs kept for analysis: {len(kept_logs)}")
print(f"Logs dropped as noise: {len(dropped_logs)}")
print(f"Storage Savings: {savings:.2f}%")

plt.figure(figsize=(10, 5))
sns.countplot(data=kept_logs, x='Attack', hue='Attack', palette='viridis', legend=False)
plt.title('Traffic Stream After Gatekeeper Filtering (Notebook B)')
plt.xticks(rotation=45)
plt.show()

kept_logs.to_csv('/content/drive/MyDrive/Colab Notebooks/kept_logs_for_training.csv', index=False)

In [ ]:
df = kept_logs.copy()
print(df.columns.tolist())
df.head()

['IPV4_SRC_ADDR', 'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE', 'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE', 'SRC_TO_DST_IAT_MIN', 'SRC_TO_DST_IAT_MAX', 'SRC_TO_DST_IAT_AVG', 'SRC_TO_DST_IAT_STDDEV', 'DST_TO_SRC_IAT_MIN', 'DST_TO_SRC_IAT_MAX', 'DST_TO_

,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,...,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack,hour_sin,hour_cos,Gatekeeper_Decision
2,59.166.0.0,47290,149.171.126.9,6881,6,37.0,13662,238,548216,438,...,119,0,1843,5,88,0,Benign,1.0,6.123234e-17,-1
5,59.166.0.1,10731,149.171.126.5,48138,6,0.0,320,6,1942,8,...,0,0,2,0,0,0,Benign,1.0,6.123234e-17,-1
7,59.166.0.1,12693,149.171.126.5,54121,6,0.0,8928,14,320,6,...,0,0,6,1,2,0,Benign,1.0,6.123234e-17,-1
9,175.45.176.0,43944,149.171.126.16,8088,6,0.0,160,4,80,2,...,355,0,0,0,0,1,Fuzzers,1.0,6.123234e-17,-1
12,59.166.0.4,13472,149.171.126.8,143,6,4.0,7816,122,15606,126,...,0,0,2,0,0,0,Benign,1.0,6.123234e-17,-1


In [ ]:
import pandas as pd
import numpy as np
import json
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_recall_curve
from kafka import KafkaProducer

# --- 1. PREPARE DATA ---
df = df.reset_index(drop=True)
BASE_PATH = "/content/drive/MyDrive/Colab Notebooks"

# Define column groups
critical_cols = ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'FLOW_DURATION_MILLISECONDS', 'IN_BYTES', 'OUT_BYTES']
important_cols = ['IN_PKTS', 'OUT_PKTS', 'TCP_FLAGS', 'SRC_TO_DST_IAT_AVG', 'DST_TO_SRC_IAT_AVG', 'MIN_TTL', 'MAX_TTL']
optional_cols = [col for col in df.columns if col not in critical_cols + important_cols + ['Label', 'Attack']]

def corrupt_logs(data):
    df_c = data.copy()
    # Simulate real-world sensor noise
    for col in np.random.choice(optional_cols, 2): df_c.loc[df_c.sample(frac=0.05).index, col] = np.nan
    for col in np.random.choice(important_cols, 3): df_c.loc[df_c.sample(frac=0.03).index, col] = np.nan
    for col in np.random.choice(critical_cols, 2): df_c.loc[df_c.sample(frac=0.01).index, col] = np.nan
    return df_c

df_corrupted = corrupt_logs(df)
df_corrupted['null_count'] = df_corrupted.isnull().sum(axis=1)

# --- 2. GATEKEEPER CALIBRATION (Random Forest) ---
features = ['IN_BYTES', 'OUT_BYTES', 'IN_PKTS', 'OUT_PKTS', 'FLOW_DURATION_MILLISECONDS', 'null_count']
X = df_corrupted[features]
y = df_corrupted['Label']

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
    ("model", RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1))
])

# Fit and find the optimized threshold for 98% recall
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipeline.fit(X_train, y_train)
y_scores = pipeline.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_scores)

target_recall = 0.98
idx = np.where(recalls >= target_recall)[0][-1]
optimized_threshold = thresholds[idx]

# --- 3. PREPARE FOR STREAMING ---
probs_all = pipeline.predict_proba(X)[:, 1]
# 'dlq' for suspicious, 'main' for normal
df_corrupted['Topic'] = np.where(probs_all >= optimized_threshold, 'dlq', 'main')
# CRITICAL: Unique ID for synchronization with Notebook 3
df_corrupted['event_id'] = range(len(df_corrupted))

# Save the Ground Truth for the Evaluator
df_corrupted.to_csv(f"{BASE_PATH}/stream_ground_truth.csv", index=False)
print(f"Ground truth saved. Threshold set at: {optimized_threshold:.4f}")

# --- 4. INITIALIZE KAFKA PRODUCER ---
producer = KafkaProducer(
    bootstrap_servers="kafka-24f52108-loginvestigation.i.aivencloud.com:27515",
    security_protocol="SSL",
    ssl_cafile=f"{BASE_PATH}/ca.pem",
    ssl_certfile=f"{BASE_PATH}/service.cert",
    ssl_keyfile=f"{BASE_PATH}/service.key",
    value_serializer=lambda v: json.dumps(v, default=str).encode("utf-8"),
    batch_size=131072,      # 128KB batches for high throughput
    linger_ms=100,          # Wait slightly to fill batches
    compression_type="gzip",# Compress to save Aiven bandwidth
    acks=1                  # Wait for lead broker acknowledgement
)

# --- 5. HIGH-SPEED STREAMING LOOP ---
main_count = 0
dlq_count = 0
start_time = time.time()

print(f"🚀 Streaming {len(df_corrupted)} records...")

for i, row in df_corrupted.iterrows():
    is_suspicious = (row["Topic"] == "dlq")
    target_topic = "dlq_logs" if is_suspicious else "main_logs"

    # Forensic Rule: Remove Label/Attack so the AI has to "guess" them
    # Keep event_id so we can verify the guess later
    record = row.drop(["Label", "Attack", "Topic"]).to_dict()

    # Handle NaNs for JSON
    clean_msg = {k: (None if pd.isna(v) else v) for k, v in record.items()}

    producer.send(target_topic, clean_msg)

    if is_suspicious: dlq_count += 1
    else: main_count += 1

    # Flush every 50k to keep memory stable
    if (i + 1) % 50000 == 0:
        print(f"Progress: {i + 1} logs queued...")
        producer.flush()

producer.flush()
producer.close()

duration = time.time() - start_time
print(f"\n--- SUCCESS ---")
print(f"Time: {duration:.2f}s | Throughput: {len(df_corrupted)/duration:.2f} logs/sec")
print(f"Sent to Main: {main_count} | Sent to DLQ: {dlq_count}")

✅ Ground truth saved. Threshold set at: 0.9100
🚀 Streaming 938936 records...
Progress: 50000 logs queued...
Progress: 100000 logs queued...
Progress: 150000 logs queued...


In [ ]:
# import pandas as pd
# import numpy as np
# import json
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.model_selection import train_test_split
# from sklearn.impute import SimpleImputer
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import classification_report, precision_recall_curve
# from kafka import KafkaProducer

# df = df.reset_index(drop=True)

# critical_cols = ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'FLOW_DURATION_MILLISECONDS', 'IN_BYTES', 'OUT_BYTES']
# important_cols = ['IN_PKTS', 'OUT_PKTS', 'TCP_FLAGS', 'SRC_TO_DST_IAT_AVG', 'DST_TO_SRC_IAT_AVG', 'MIN_TTL', 'MAX_TTL']
# optional_cols = [col for col in df.columns if col not in critical_cols + important_cols + ['Label', 'Attack']]

# def corrupt_logs(df):
#     df_corrupted = df.copy()
#     for col in np.random.choice(optional_cols, 2): df_corrupted.loc[df_corrupted.sample(frac=0.05).index, col] = np.nan
#     for col in np.random.choice(important_cols, 3): df_corrupted.loc[df_corrupted.sample(frac=0.03).index, col] = np.nan
#     for col in np.random.choice(critical_cols, 2): df_corrupted.loc[df_corrupted.sample(frac=0.01).index, col] = np.nan
#     return df_corrupted

# df_corrupted = corrupt_logs(df)

# df_corrupted['null_count'] = df_corrupted.isnull().sum(axis=1)

# features = [
#     'IN_BYTES', 'OUT_BYTES', 'IN_PKTS', 'OUT_PKTS',
#     'FLOW_DURATION_MILLISECONDS', 'null_count'
# ]

# X = df_corrupted[features]
# y = df_corrupted['Label']

# pipeline = Pipeline([
#     ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
#     ("model", RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1))
# ])

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# pipeline.fit(X_train, y_train)
# y_scores_test = pipeline.predict_proba(X_test)[:, 1]
# precisions, recalls, thresholds = precision_recall_curve(y_test, y_scores_test)

# target_recall = 0.98
# idx = np.where(recalls >= target_recall)[0][-1]
# optimized_threshold = thresholds[idx]

# print(f"--- Calibration ---")
# print(f"To catch {target_recall*100}% of attacks, using Threshold: {optimized_threshold:.4f}")

# probs_all = pipeline.predict_proba(X)[:, 1]
# df_corrupted['Topic'] = np.where(probs_all >= optimized_threshold, 'dlq', 'main')

# df_corrupted['event_id'] = range(len(df_corrupted))

# df_corrupted.to_csv('/content/drive/MyDrive/Colab Notebooks/stream_ground_truth.csv', index=False)
# print("Ground truth with event_id saved to Drive.")


# def stream_data(dataframe):
#     print(f"🚀 Starting stream of {len(dataframe)} logs...")

#     for _, row in dataframe.iterrows():
#         topic_flag = row['Topic']
#         target_topic = "dlq_logs" if topic_flag == 'dlq' else "main_logs"
#         msg = row.drop('Topic').to_dict()
#         msg = {k: (None if pd.isna(v) else v) for k, v in msg.items()}
#         producer.send(target_topic, value=msg)

#     producer.flush()
#     print("Streaming simulation complete.")

# stream_data(df_corrupted)

KeyboardInterrupt: 

In [ ]:
# from kafka import KafkaProducer
# import json
# import time

# def on_send_success(record_metadata):
#     pass

# def on_send_error(excp):
#     print(f"FAILED to send message: {excp}")

# producer = KafkaProducer(
#     bootstrap_servers="kafka-24f52108-loginvestigation.i.aivencloud.com:27515",
#     security_protocol="SSL",
#     ssl_cafile="/content/drive/MyDrive/Colab Notebooks/ca.pem",
#     ssl_certfile="/content/drive/MyDrive/Colab Notebooks/service.cert",
#     ssl_keyfile="/content/drive/MyDrive/Colab Notebooks/service.key",
#     value_serializer=lambda v: json.dumps(v, default=str).encode("utf-8"),
#     api_version_auto_timeout_ms=30000,
#     linger_ms=100,
#     batch_size=131072,
#     compression_type="gzip",
#     request_timeout_ms=60000,
#     acks=1,
#     retries=3

# )

# print("Connected to Aiven Kafka successfully!")

# MAIN_TOPIC = "main_logs"
# DLQ_TOPIC = "dlq_logs"

# main_count = 0
# dlq_count = 0
# def clean_record(d):
#     return {k: (None if pd.isna(v) else v) for k, v in d.items()}

# stream_limit = len(df_corrupted)                #stream limit
# print(f"Streaming {stream_limit} records...")

# start_time = time.time()

# for i, row in df_corrupted.iterrows():
#     is_suspicious = (row["Topic"] != "main")
#     target_topic = DLQ_TOPIC if is_suspicious else MAIN_TOPIC
#     drop_cols = ["Label", "Attack", "Topic"]
#     if not is_suspicious:
#         drop_cols.append("null_count")

#     record = row.drop(drop_cols).to_dict()
#     clean_msg = clean_record(record)
#     producer.send(target_topic, clean_msg).add_callback(on_send_success).add_errback(on_send_error)
#     if is_suspicious:
#         dlq_count += 1
#     else:
#         main_count += 1

#     if (i + 1) % 50000 == 0:
#         print(f"Progress: {i + 1} rows queued... Clearing buffer...")
#         producer.flush()
# producer.flush()
# producer.close()

# end_time = time.time()
# duration = end_time - start_time

# print("\n--- Streaming Complete! ---")
# print(f"Total Time: {duration:.2f} seconds")
# print(f"Sent to {MAIN_TOPIC}: {main_count}")
# print(f"Sent to {DLQ_TOPIC}: {dlq_count}")
# print(f"Throughput: {stream_limit/duration:.2f} logs/sec")



In [ ]:
# producer.send("dlq_logs", {"test": "hello from colab"})
# producer.flush()

# print("Test message sent!")
